## Generate Embeddings (text-embedding-3-small, capped PDF sample)

Generates embeddings using `text-embedding-3-small` (via `langchain-openai`) as a second candidate model alongside the earlier `text-embedding-3-large` run, for the eventual embedding-model benchmark comparison.

**Cost control for this run**: abstracts are embedded in full (cheap), but PDF chunks are capped at the first 100 papers per category (by `paper_id`, sorted for determinism), not the first 100 page-chunks. Since each paper contributes multiple page-chunks, this still means several hundred embedded chunks per category, not literally 100, but it avoids the cost of embedding every page of every one of the 500 papers per category. Increase `MAX_PDFS_PER_CATEGORY` later once you're ready to embed the full PDF corpus with this model.

**What gets embedded**:
- Abstracts: the `text` field, full corpus, all categories
- PDF chunks: the `text` field, first `MAX_PDFS_PER_CATEGORY` papers per category only (all pages of those papers included)

**Resumability**: if an embedding output file for a category already exists locally, it's skipped unless `FORCE_REGENERATE = True`.

**Storage note**: same as before, embeddings stored as JSONL (`chunk_id` + `embedding` list). `text-embedding-3-small` defaults to 1536 dimensions, half the size of `text-embedding-3-large`'s 3072, so files here will be smaller.

### Install dependencies (Colab only, skip if already installed locally)

In [ ]:
# Uncomment if running in Colab or a fresh environment
# !pip install langchain-openai huggingface_hub python-dotenv tqdm -q

### Configuration

In [1]:
import os

# --- Hugging Face repo (shared corpus) ---
REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"
REPO_TYPE = "dataset"

# --- Embedding model ---
EMBEDDING_MODEL = "text-embedding-3-small"

# --- The 10 target categories ---
ALL_CATEGORIES = [
    "cs.AI", "cs.LG", "cs.IR", "cs.DB", "cs.SE",       # yours
    "cs.CV", "cs.CL", "cs.NE", "cs.DC", "cs.CR",       # collaborator's
]

# --- Local output paths ---
PROJECT_ROOT = ".."
EMBEDDINGS_DIR = os.path.join(PROJECT_ROOT, "data", "embeddings", EMBEDDING_MODEL)
os.makedirs(os.path.join(EMBEDDINGS_DIR, "abstracts"), exist_ok=True)
os.makedirs(os.path.join(EMBEDDINGS_DIR, "pdf_chunks"), exist_ok=True)

# --- Behavior ---
FORCE_REGENERATE = False  # set True to re-embed categories that already have output files
BATCH_SIZE = 100          # chunks per API call, OpenAI allows large batches for embeddings
REQUEST_DELAY_SECONDS = 0.5  # small delay between batches to stay comfortably under rate limits
MAX_PDFS_PER_CATEGORY = 100  # cap on distinct paper_ids sampled per category for PDF chunks, cost control

### Authenticate (OpenAI + Hugging Face)

Requires `OPENAI_API_KEY` in your environment or a `.env` file (loaded via `python-dotenv`, matching the pattern in your existing question-generation notebook). Also confirms Hugging Face auth for pulling the source corpus.

In [2]:
from dotenv import load_dotenv
from huggingface_hub import whoami, login

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found, set it in your .env file or environment"

try:
    user_info = whoami()
    print(f"Hugging Face authenticated as: {user_info['name']}")
except Exception:
    print("Not authenticated with Hugging Face yet, prompting for token...")
    login()
    user_info = whoami()
    print(f"Authenticated as: {user_info['name']}")

c:\project\empirical-rag-paradigm-benchmark\benchmarking-research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face authenticated as: Sakhiur


### Set up the embedder

`OpenAIEmbeddings` from `langchain-openai` batches multiple texts per API call automatically when you pass a list to `embed_documents`, so we don't need to hand-roll batching logic beyond chunking our input list into reasonably sized pieces (`BATCH_SIZE`) to keep memory and retry scope manageable.

In [3]:
from langchain_openai import OpenAIEmbeddings

embedder = OpenAIEmbeddings(model=EMBEDDING_MODEL)

### Helper functions

- `load_category_records`: pulls a category's JSONL from the Hugging Face repo (abstracts or PDF chunks) and parses it.
- `embed_records`: batches texts through the embedder, with retry-on-failure so one bad batch doesn't kill the whole run.
- `save_embeddings`: writes `chunk_id` + `embedding` pairs to a local JSONL file.

In [4]:
import json
import time
from huggingface_hub import hf_hub_download


def load_category_records(category: str, corpus_type: str) -> list[dict]:
    """
    corpus_type: 'abstracts' or 'pdf_chunks'
    Returns a list of parsed JSON records, or an empty list if the file doesn't exist on the Hub
    (expected for categories your collaborator hasn't uploaded yet).
    """
    if corpus_type == "abstracts":
        repo_path = f"abstracts/by_category/{category}.jsonl"
    elif corpus_type == "pdf_chunks":
        repo_path = f"pdf_chunks/{category}.jsonl"
    else:
        raise ValueError(f"Unknown corpus_type: {corpus_type}")

    try:
        local_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=repo_path)
    except Exception as e:
        print(f"  [SKIP] {repo_path} not found on Hub ({e})")
        return []

    records = []
    with open(local_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"  [WARN] {repo_path} line {line_num}: malformed JSON, skipping")
    return records


def get_chunk_id_and_text(record: dict, corpus_type: str) -> tuple[str, str]:
    """Extracts (chunk_id, text_to_embed) from a record, per corpus_type's schema."""
    if corpus_type == "abstracts":
        return record["id"], record["text"]
    elif corpus_type == "pdf_chunks":
        chunk_id = f"{record['paper_id']}_p{record['page_num']}"
        return chunk_id, record["text"]
    else:
        raise ValueError(f"Unknown corpus_type: {corpus_type}")


def embed_records(records: list[dict], corpus_type: str, max_retries: int = 3) -> list[dict]:
    """Embeds a list of records in batches, returns list of {chunk_id, embedding} dicts."""
    results = []

    chunk_ids = []
    texts = []
    for record in records:
        chunk_id, text = get_chunk_id_and_text(record, corpus_type)
        if not text or not text.strip():
            print(f"  [WARN] {chunk_id}: empty text, skipping")
            continue
        chunk_ids.append(chunk_id)
        texts.append(text)

    for batch_start in range(0, len(texts), BATCH_SIZE):
        batch_ids = chunk_ids[batch_start : batch_start + BATCH_SIZE]
        batch_texts = texts[batch_start : batch_start + BATCH_SIZE]

        for attempt in range(1, max_retries + 1):
            try:
                batch_embeddings = embedder.embed_documents(batch_texts)
                break
            except Exception as e:
                print(f"  [WARN] batch {batch_start}-{batch_start+len(batch_texts)}: attempt {attempt} failed ({e})")
                if attempt == max_retries:
                    print(f"  [ERROR] batch {batch_start}-{batch_start+len(batch_texts)}: giving up after {max_retries} attempts")
                    batch_embeddings = [None] * len(batch_texts)
                else:
                    time.sleep(2 ** attempt)  # exponential backoff

        for cid, emb in zip(batch_ids, batch_embeddings):
            if emb is not None:
                results.append({"chunk_id": cid, "embedding": emb})

        time.sleep(REQUEST_DELAY_SECONDS)

    return results


def save_embeddings(embeddings: list[dict], out_path: str) -> None:
    with open(out_path, "w", encoding="utf-8") as f:
        for record in embeddings:
            f.write(json.dumps(record) + "\n")


def cap_pdf_records_by_paper(records: list[dict], max_papers: int) -> list[dict]:
    """
    Keeps all page-chunks belonging to the first `max_papers` distinct paper_ids,
    sorted for determinism (so re-runs pick the same 100 papers rather than an
    arbitrary/unstable subset depending on file iteration order).
    Returns all pages for each selected paper, not a flat cap on chunk count.
    """
    unique_paper_ids = sorted({r["paper_id"] for r in records})
    selected_paper_ids = set(unique_paper_ids[:max_papers])
    filtered = [r for r in records if r["paper_id"] in selected_paper_ids]
    return filtered

### Generate embeddings: abstracts

Loops all 10 categories. Categories not yet uploaded by your collaborator are skipped automatically (reported, not treated as an error). Already-embedded categories are skipped unless `FORCE_REGENERATE = True`.

In [6]:
from tqdm import tqdm

abstract_summary = {}

for category in ALL_CATEGORIES:
    out_path = os.path.join(EMBEDDINGS_DIR, "abstracts", f"{category}.jsonl")

    if os.path.exists(out_path) and not FORCE_REGENERATE:
        print(f"[SKIP] {category} abstracts: already embedded at {out_path}")
        continue

    print(f"\n=== {category} abstracts ===")
    records = load_category_records(category, "abstracts")
    if not records:
        abstract_summary[category] = 0
        continue

    embeddings = embed_records(records, "abstracts")
    save_embeddings(embeddings, out_path)
    abstract_summary[category] = len(embeddings)
    print(f"  {len(embeddings)}/{len(records)} embedded -> {out_path}")

print("\nAbstract embedding summary:")
for category, count in abstract_summary.items():
    print(f"  {category}: {count}")

[SKIP] cs.AI abstracts: already embedded at ..\data\embeddings\text-embedding-3-small\abstracts\cs.AI.jsonl
[SKIP] cs.LG abstracts: already embedded at ..\data\embeddings\text-embedding-3-small\abstracts\cs.LG.jsonl
[SKIP] cs.IR abstracts: already embedded at ..\data\embeddings\text-embedding-3-small\abstracts\cs.IR.jsonl
[SKIP] cs.DB abstracts: already embedded at ..\data\embeddings\text-embedding-3-small\abstracts\cs.DB.jsonl
[SKIP] cs.SE abstracts: already embedded at ..\data\embeddings\text-embedding-3-small\abstracts\cs.SE.jsonl
[SKIP] cs.CV abstracts: already embedded at ..\data\embeddings\text-embedding-3-small\abstracts\cs.CV.jsonl
[SKIP] cs.CL abstracts: already embedded at ..\data\embeddings\text-embedding-3-small\abstracts\cs.CL.jsonl

=== cs.NE abstracts ===
  1501/1501 embedded -> ..\data\embeddings\text-embedding-3-small\abstracts\cs.NE.jsonl

=== cs.DC abstracts ===
  1517/1517 embedded -> ..\data\embeddings\text-embedding-3-small\abstracts\cs.DC.jsonl

=== cs.CR abstrac

### Generate embeddings: PDF chunks (capped sample)

Same pattern as abstracts, but each category's records are first filtered down to the first `MAX_PDFS_PER_CATEGORY` distinct papers (by `paper_id`, sorted) before embedding. All page-chunks for those selected papers are kept, so the actual embedded chunk count will be higher than `MAX_PDFS_PER_CATEGORY`, that's expected.

In [7]:
pdf_summary = {}

for category in ALL_CATEGORIES:
    out_path = os.path.join(EMBEDDINGS_DIR, "pdf_chunks", f"{category}.jsonl")

    if os.path.exists(out_path) and not FORCE_REGENERATE:
        print(f"[SKIP] {category} PDF chunks: already embedded at {out_path}")
        continue

    print(f"\n=== {category} PDF chunks (capped at {MAX_PDFS_PER_CATEGORY} papers) ===")
    all_records = load_category_records(category, "pdf_chunks")
    if not all_records:
        pdf_summary[category] = 0
        continue

    total_papers_available = len({r["paper_id"] for r in all_records})
    records = cap_pdf_records_by_paper(all_records, MAX_PDFS_PER_CATEGORY)
    papers_selected = len({r["paper_id"] for r in records})
    print(f"  {papers_selected}/{total_papers_available} papers selected, {len(records)} page-chunks to embed")

    embeddings = embed_records(records, "pdf_chunks")
    save_embeddings(embeddings, out_path)
    pdf_summary[category] = len(embeddings)
    print(f"  {len(embeddings)}/{len(records)} embedded, saved to {out_path}")

print("\nPDF chunk embedding summary:")
for category, count in pdf_summary.items():
    print(f"  {category}: {count}")



=== cs.AI PDF chunks (capped at 100 papers) ===
  100/498 papers selected, 2062 page-chunks to embed
  [WARN] 2101.00058_p38: empty text, skipping
  [WARN] 2101.00058_p39: empty text, skipping
  2060/2062 embedded, saved to ..\data\embeddings\text-embedding-3-small\pdf_chunks\cs.AI.jsonl

=== cs.LG PDF chunks (capped at 100 papers) ===
  100/498 papers selected, 1937 page-chunks to embed
  1937/1937 embedded, saved to ..\data\embeddings\text-embedding-3-small\pdf_chunks\cs.LG.jsonl

=== cs.IR PDF chunks (capped at 100 papers) ===
  100/499 papers selected, 1371 page-chunks to embed
  1371/1371 embedded, saved to ..\data\embeddings\text-embedding-3-small\pdf_chunks\cs.IR.jsonl

=== cs.DB PDF chunks (capped at 100 papers) ===
  100/498 papers selected, 1768 page-chunks to embed
  1768/1768 embedded, saved to ..\data\embeddings\text-embedding-3-small\pdf_chunks\cs.DB.jsonl

=== cs.SE PDF chunks (capped at 100 papers) ===
  100/500 papers selected, 1956 page-chunks to embed
  1956/1956 em

### Upload embeddings back to Hugging Face

Stored under `embeddings/<model_name>/abstracts/<category>.jsonl` and `embeddings/<model_name>/pdf_chunks/<category>.jsonl`, so the model used is part of the path, this matters once you generate embeddings from a second model for the actual benchmark comparison later.

In [8]:
from huggingface_hub import HfApi

api = HfApi()

for corpus_type in ["abstracts", "pdf_chunks"]:
    local_dir = os.path.join(EMBEDDINGS_DIR, corpus_type)
    for fname in sorted(os.listdir(local_dir)):
        if not fname.endswith(".jsonl"):
            continue
        local_path = os.path.join(local_dir, fname)
        repo_path = f"embeddings/{EMBEDDING_MODEL}/{corpus_type}/{fname}"

        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            commit_message=f"Upload {EMBEDDING_MODEL} embeddings for {corpus_type}/{fname} (by {user_info['name']})",
        )
        print(f"Uploaded {repo_path}")

Processing Files (1 / 1): 100%|██████████|  229MB /  229MB, 4.30MB/s  
New Data Upload: 100%|██████████|  229MB /  229MB, 4.30MB/s  


Uploaded embeddings/text-embedding-3-small/abstracts/cs.AI.jsonl


Processing Files (1 / 1): 100%|██████████|  385MB /  385MB, 7.20MB/s  
New Data Upload: 100%|██████████|  385MB /  385MB, 7.20MB/s  


Uploaded embeddings/text-embedding-3-small/abstracts/cs.CL.jsonl


Processing Files (1 / 1): 100%|██████████| 47.2MB / 47.2MB, 2.80MB/s  
New Data Upload: 100%|██████████| 47.2MB / 47.2MB, 2.80MB/s  


...

Uploaded embeddings/text-embedding-3-small/pdf_chunks/cs.LG.jsonl


Processing Files (1 / 1): 100%|██████████| 51.8MB / 51.8MB, 3.20MB/s  
New Data Upload: 100%|██████████| 51.8MB / 51.8MB, 3.20MB/s  


Uploaded embeddings/text-embedding-3-small/pdf_chunks/cs.NE.jsonl


Processing Files (1 / 1): 100%|██████████| 60.7MB / 60.7MB, 3.48MB/s  
New Data Upload: 100%|██████████| 60.7MB / 60.7MB, 3.48MB/s  


Uploaded embeddings/text-embedding-3-small/pdf_chunks/cs.SE.jsonl


### Verify

Confirms dimensionality is consistent (should all be the same for a given model) and reports final counts per category, across both corpora, so you can see at a glance what's ready for clustering versus what's still missing (your collaborator's categories, most likely).

In [9]:
print(f"Model: {EMBEDDING_MODEL}\n")

for corpus_type, summary in [("abstracts", abstract_summary), ("pdf_chunks", pdf_summary)]:
    print(f"{corpus_type}:")
    for category in ALL_CATEGORIES:
        out_path = os.path.join(EMBEDDINGS_DIR, corpus_type, f"{category}.jsonl")
        if not os.path.exists(out_path):
            print(f"  {category}: not embedded (missing source or skipped)")
            continue
        with open(out_path, "r", encoding="utf-8") as f:
            lines = [json.loads(l) for l in f if l.strip()]
        dims = {len(r["embedding"]) for r in lines}
        dim_status = dims.pop() if len(dims) == 1 else f"INCONSISTENT: {dims}"
        print(f"  {category}: {len(lines)} vectors, dim={dim_status}")
    print()

Model: text-embedding-3-small

abstracts:
  cs.AI: 7369 vectors, dim=1536
  cs.LG: 10746 vectors, dim=1536
  cs.IR: 3076 vectors, dim=1536
  cs.DB: 1532 vectors, dim=1536
  cs.SE: 2281 vectors, dim=1536
  cs.CV: 14163 vectors, dim=1536
  cs.CL: 12418 vectors, dim=1536
  cs.NE: 1501 vectors, dim=1536
  cs.DC: 1517 vectors, dim=1536
  cs.CR: 1523 vectors, dim=1536

pdf_chunks:
  cs.AI: 2060 vectors, dim=1536
  cs.LG: 1937 vectors, dim=1536
  cs.IR: 1371 vectors, dim=1536
  cs.DB: 1768 vectors, dim=1536
  cs.SE: 1956 vectors, dim=1536
  cs.CV: 1250 vectors, dim=1536
  cs.CL: 1363 vectors, dim=1536
  cs.NE: 1668 vectors, dim=1536
  cs.DC: 1826 vectors, dim=1536
  cs.CR: 1517 vectors, dim=1536

